# L13d Lab: An MCP Server for the FRED API
Previously, we built a software development kit (SDK) for three endpoints of the FRED API, and downloaded the price of natural gas. A language model application could answer questions about prices if it could call FRED through that SDK. For example, a process engineer could ask the application: "Was natural gas cheaper in 2025 than in 2024?"

We'll explore this question using a Model Context Protocol (MCP) server that wraps the FRED SDK. The server offers three tools. You will complete one of the tools, and then use the server to answer the question as a host would.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Use the tools of an MCP server:__ A host starts the server, lists its tools, and calls a tool with arguments that match its input schema. We'll start the FRED server, list its three tools, and search FRED for a series.
> * __Write an input schema and a tool handler:__ The input schema tells a language model which arguments a tool takes, and the handler runs the tool. You will write the input schema and the handler of the observations tool.
> * __Answer a question as the host:__ A language model picks a tool and its arguments, and the host runs the call. We'll pick the tool and the arguments that compare the price of natural gas in 2024 and 2025, and read the errors that wrong arguments produce.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

In addition to the Julia standard libraries, we use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), [JSON.jl](https://github.com/JuliaIO/JSON.jl) to print the messages, and [DataFrames.jl](https://dataframes.juliadata.org/stable/) with [PrettyTables.jl](https://ronisbr.github.io/PrettyTables.jl/stable/) for the tables.

The code of the FRED server is in the [src](src) folder. The first four files are our FRED SDK:

* [src/Types.jl](src/Types.jl) holds the endpoint models, and the models of an MCP tool, server, and client.
* [src/Factory.jl](src/Factory.jl) holds the factory methods that build the request URLs, the tools, and the server.
* [src/Network.jl](src/Network.jl) holds the functions that make the HTTP calls.
* [src/Handler.jl](src/Handler.jl) holds the handlers, the functions that process a response.
* [src/Utility.jl](src/Utility.jl) holds the function that reads your API key from the configuration file.
* [src/Tools.jl](src/Tools.jl) holds the three tools of the FRED server. You will complete one tool in this file.
* [src/Server.jl](src/Server.jl) and [src/Client.jl](src/Client.jl) hold the MCP server loop and the client functions, the same code as in the weather server example.

The [server.jl](server.jl) file is the program that a host starts. It loads the code in the [src](src) folder, reads your API key, builds the FRED server, and runs the server loop.

### API key
The FRED server needs your FRED API key, the key that you used to build the FRED SDK. To store the key for this lab:

1. Copy your `Configuration.toml` file from the `L13b/conf` folder of the FRED SDK lab into the [conf](conf) folder of this lab. If you do not have that file, copy [conf/Configuration-template.toml](conf/Configuration-template.toml) to a new file named `Configuration.toml` in the same folder.
2. Check that your key is between the quotes on the `apikey` line, and save the file.

The server reads the key when it starts, and puts the key into its HTTPS requests to FRED. The key never appears in an MCP message, so a language model never sees your key. This notebook never reads the key either.

### Constants
We'll set the command that starts the FRED server. The command runs [server.jl](server.jl) in the course environment, with the Julia program that runs this notebook:

In [ ]:
julia_program = joinpath(Sys.BINDIR, "julia"); # the Julia program that runs this notebook
server_command = `$(julia_program) --startup-file=no --project=$(CHEME5800_ROOT) $(CHEME5800_L13D_SERVER)`; # the command that starts the FRED server
println("Configuration file found: $(isfile(joinpath(CHEME5800_L13D_CONFIG, "Configuration.toml")))"); # true once you copied the file

___
## Task 1: Start the Server and Search FRED
In this task, we start the FRED server, list its tools, and search FRED for the series that holds the price of natural gas.

Let's start the server with [the `connect(...)` function](src/Client.jl), and send a `server/discover` request. The server loads its packages before it reads its first message, so the first response takes a few seconds:

In [ ]:
client = connect(server_command); # start the server, and connect to it
discover_response = discover(client); # send a server/discover request, and read the response
discover_result = discover_response["result"];
println("Server: ", discover_result["_meta"]["io.modelcontextprotocol/serverInfo"]["name"]);
println("Protocol versions: ", join(discover_result["supportedVersions"], ", "));
println("Instructions: ", discover_result["instructions"]);

The instructions tell a language model how to use the server: search for the identifier of a series first, and then download its values.

Next, let's list the tools, and put the name, the required arguments, and the description of each tool into a table:

In [ ]:
list_response = listtools(client); # send a tools/list request, and read the response
tools = list_response["result"]["tools"]; # one dictionary for each tool

# one row for each tool -
tool_table = DataFrame(name = String[], required = String[], description = String[]);
for tool ∈ tools
    required_arguments = join(tool["inputSchema"]["required"], ", "); # the arguments that every call must give
    push!(tool_table, (tool["name"], required_arguments, tool["description"]));
end
pretty_table(tool_table; backend = :text, display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact));

__What do we see?__ The server has three tools. The `get_series_observations` tool has no required arguments yet, because its input schema is empty until you complete TODO 1 in Task 2.

Now, let's call the `search_series` tool with [the `calltool(...)` function](src/Client.jl). A language model would write these arguments from the input schema of the tool:

In [ ]:
search_arguments = Dict{String,Any}("search_text" => "Henry Hub natural gas spot price", "limit" => 5);
search_response = calltool(client, "search_series", search_arguments);
search_result = search_response["result"];

# show the result. A tool execution error has no structuredContent, so we print its text instead -
if (search_result["isError"] == true)
    println(search_result["content"][1]["text"]); # the reason the tool failed, e.g., a missing configuration file
else
    series_table = DataFrame(series_id = String[], title = String[], frequency = String[]);
    for series ∈ search_result["structuredContent"]["series"]
        push!(series_table, (series["series_id"], series["title"], series["frequency"]));
    end
    pretty_table(series_table; backend = :text, display_size = (-1, -1),
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

FRED holds the Henry Hub price at more than one frequency. We'll use the daily series, which has the identifier `DHHNGSP`. If the call printed a message about the configuration file, fix the file as described in the Setup section, and run this task again.

___
## Task 2: Complete the Observations Tool
In this task, you will complete the `get_series_observations` tool. The tool downloads the values of one series between two dates.

A tool has two parts that you write. The input schema tells a host which arguments the tool takes, and the handler is the function that the server runs for each call.

A TODO comment in [src/Tools.jl](src/Tools.jl) marks each part:

> __Implementation tasks__
>
> * __TODO 1:__ In [the `build_fred_server(...)` function](src/Tools.jl) at the bottom of `src/Tools.jl`, fill in the input schema of the observations tool. The tool takes four text arguments: `series_id`, `observation_start`, `observation_end`, and `frequency`. The first three are required.
> * __TODO 2:__ In [the `get_series_observations(...)` function](src/Tools.jl), download the observations as a table with the SDK. Use [the `MyFredSeriesObservationsEndpointModel` type](src/Types.jl) and [the `process_observations_response_dataframe(...)` function](src/Handler.jl) as the handler.
> * __TODO 3:__ In the same function, put the date and the value of each row of the table into a dictionary.

The server loads [src/Tools.jl](src/Tools.jl) each time it starts. After each edit, save the file, and run the cells of this task again. You do not need to restart the kernel.

Let's close the server, and start a new one that loads your edits:

In [ ]:
close(client); # stop the server that loaded the old file
client = connect(server_command); # start a new server, which loads your src/Tools.jl

Now, let's ask for the monthly price of natural gas in 2025. With `frequency = "m"`, FRED averages the daily prices of each month. We print the text of the result, which a host gives to its language model:

In [ ]:
observation_arguments = Dict{String,Any}("series_id" => "DHHNGSP", "observation_start" => "2025-01-01",
    "observation_end" => "2025-12-31", "frequency" => "m");
observation_response = calltool(client, "get_series_observations", observation_arguments);
println(observation_response["result"]["content"][1]["text"]) # the text that a host gives to its model

The untouched file returns a tool execution error with the message `Oooops!`. Once your handler works, the text holds one date and one value for each month.

### Check: Observations tool
The check lists the tools again, reads your input schema, and calls your tool for the twelve months of 2025:

In [ ]:
@testset "Observations tool" begin

    # TODO 1: the input schema of the observations tool -
    observations_tool = nothing;
    for tool ∈ listtools(client)["result"]["tools"]
        if (tool["name"] == "get_series_observations")
            observations_tool = tool;
        end
    end
    schema = observations_tool["inputSchema"];
    @test Set(keys(schema["properties"])) == Set(["series_id", "observation_start", "observation_end", "frequency"])
    @test Set(schema["required"]) == Set(["series_id", "observation_start", "observation_end"])
    for (name, property) ∈ schema["properties"]
        @test property["type"] == "string" # every argument is text
    end

    # TODO 2 and TODO 3: the call returns one observation for each month of 2025 -
    result = observation_response["result"];
    @test result["isError"] == false
    observations = result["structuredContent"]["observations"];
    @test length(observations) == 12
    @test observations[1]["date"] == "2025-01-01" # FRED dates each monthly average on the first day of the month
    for observation ∈ observations
        @test observation["value"] isa Real # every month of 2025 has an average price
    end
end;

___
## Task 3: Answer a Question as the Host
In this task, we answer the question of the process engineer: was natural gas cheaper in 2025 than in 2024?

A language model would read the input schemas, and pick a tool and its arguments. We make the same choice here. The `get_series_observations` tool with `frequency = "a"` returns one average price per year, so one call answers the question:

In [ ]:
annual_arguments = Dict{String,Any}("series_id" => "DHHNGSP", "observation_start" => "2024-01-01",
    "observation_end" => "2025-12-31", "frequency" => "a");
annual_response = calltool(client, "get_series_observations", annual_arguments);
annual_observations = annual_response["result"]["structuredContent"]["observations"]; # one observation per year

# one row for each year (units of the price: dollars per million BTU) -
annual_table = DataFrame(year = String[], average_price = Float64[]);
for observation ∈ annual_observations
    year = observation["date"][1:4]; # the first four characters of the date
    push!(annual_table, (year, observation["value"]));
end
pretty_table(annual_table; backend = :text, display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact));

A language model would turn this result into an answer for the user. Let's compute the change in the average price from 2024 to 2025:

In [ ]:
price_2024 = annual_table[1, :average_price]; # dollars per million BTU
price_2025 = annual_table[2, :average_price]; # dollars per million BTU
percent_change = 100 * (price_2025 - price_2024) / price_2024;
println("The average price changed from $(round(price_2024, digits = 2)) to $(round(price_2025, digits = 2)) dollars per million BTU, a change of $(round(percent_change, digits = 1))%.");

__Was natural gas cheaper in 2025?__ No. The average Henry Hub price was higher in 2025 than in 2024.

### Wrong arguments
A language model writes the arguments of each call, so the arguments can be wrong. Let's send three calls with wrong arguments: a series identifier with extra text that would change the URL, a date that does not exist, and a request for more than ten years of daily prices:

In [ ]:
wrong_calls = [
    Dict{String,Any}("series_id" => "DHHNGSP&units=pch", "observation_start" => "2025-01-01", "observation_end" => "2025-12-31"),
    Dict{String,Any}("series_id" => "DHHNGSP", "observation_start" => "2025-13-01", "observation_end" => "2025-12-31"),
    Dict{String,Any}("series_id" => "DHHNGSP", "observation_start" => "2015-01-01", "observation_end" => "2025-12-31", "frequency" => "d"),
];
for arguments ∈ wrong_calls
    response = calltool(client, "get_series_observations", arguments);
    println("isError = $(response["result"]["isError"]): ", response["result"]["content"][1]["text"]);
end

__What do we see?__ All three calls are tool execution errors, so a host gives each message to its model, and the model can try again:

* The tool checks the series identifier before it builds the URL. The text `&units=pch` would have added a query parameter to the request.
* The tool checks that each date is a real date.
* The tool returns at most 400 observations, so that a result stays short enough for a language model to read. The message tells the model to ask for a lower frequency.

### Close the Connection
We are done with the server. Closing the connection ends the server loop, and the server exits:

In [ ]:
close(client); # close the standard input of the server, and wait for the server to exit

> __Connect a host to the FRED server:__ A host such as Claude Code starts the server with the same command. For example, the command below registers the FRED server with Claude Code, where `/path/to/L13d/server.jl` is the path of the [server.jl](server.jl) file on your computer:
>
> `claude mcp add fred -- julia --startup-file=no /path/to/L13d/server.jl`
>
> You can then ask Claude Code the question of the process engineer. Claude Code asks for your approval before its model calls a tool, unless your permission settings already allow the call.

___
## Summary
In this lab, we used an MCP server that wraps the FRED SDK, completed one of its tools, and answered a question about the price of natural gas as a host would.

> __Key Takeaways:__
>
> * __The tools of an MCP server:__ The FRED server turned three endpoints of the SDK into three tools, each with a name, a description, and an input schema. The server kept the API key in its own process, so no MCP message carried the key.
> * __An input schema and a tool handler:__ Your input schema told a host which arguments the observations tool takes, and which of them are required. Your handler downloaded the observations with the SDK, and returned each one as a date and a value.
> * __The host and the arguments:__ We picked the tool and the arguments that compare the price of natural gas in 2024 and 2025, as a language model would. Wrong arguments came back as tool execution errors, with a reason that a language model can read.

We can now give a language model access to any series in FRED, with no code written for that model.
___